In [ ]:
"""
weights_sess.ipynb

Plot beta weight distribution across sessions

Author: Stellina X. Ao
Created: 2026-06-30
Last Modified: 2026-06-30
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

# weight distro across sessions

In [ ]:
import numpy as np
from core.data import subject_ids, session_ids
from sg.models import Encoder

sess_ids = session_ids[np.where(subject_ids == subj_id)[0][0]]
coefs_sess = {"all": [], "DLS": [], "DMS": []}
regressors = np.array(
    ["response_left", "response_right", "rewarded_incorr", "rewarded_corr"]
)

for sess_id in sess_ids:
    encoder = Encoder(subj_id, sess_id, num_tents=5)
    encoder.fit_encoder()

    # get coefs and separate by region as well
    coefs = encoder.encoder.coef_

    tv_idxs = [i for i, dm_name in enumerate(encoder.dm_names) if dm_name in regressors]

    coefs_ = coefs[:, tv_idxs]

    coefs_sess["all"].append(coefs_)
    coefs_sess["DLS"].append(coefs_[encoder.reg_idxs["DLS"]])
    coefs_sess["DMS"].append(coefs_[encoder.reg_idxs["DMS"]])

In [ ]:
from core.viz import plot_boxplots_sess
from core.utils import beta_str


def get_coefs_sess_regr(coefs_sess, reg="all", regr="response", val="left"):
    regr_idx = np.where(regressors == f"{regr}_{val}")
    coefs_regr = [np.squeeze(coef_sess_[:, regr_idx]) for coef_sess_ in coefs_sess[reg]]
    return coefs_regr


regr = "response"
val = "left"

coefs_regr = get_coefs_sess_regr(coefs_sess, reg="all", regr=regr, val=val)
plot_boxplots_sess(
    data_sess=coefs_regr, sess_ids=sess_ids, data_label=beta_str(regr, val)
)

In [ ]:
from core.viz import plot_ridges_sess
import pandas as pd


def coefs_to_dict(coefs_sess, sess_ids):
    dfs = []
    for i, sess_id in enumerate(sess_ids):
        df = pd.DataFrame(coefs_sess["all"][i], columns=regressors)
        df.insert(0, "key", sess_id)
        dfs.append(df)

    coefs_df = pd.concat(dfs, ignore_index=True)
    return coefs_df


regr = "response"
val = "left"

coefs_sess_dict = coefs_to_dict(coefs_sess, sess_ids)
plot_ridges_sess(
    data_sess=coefs_sess_dict,
    key=f"{regr}_{val}",
    sess_ids=sess_ids,
    data_label=beta_str(regr, val),
)

### mb v. mf

In [ ]:
import numpy as np
from core.data import subject_ids, session_ids
from sg.models import StrategyEncoder

sess_ids = session_ids[np.where(subject_ids == subj_id)[0][0]]
sess_ids_ = []
coefs_sess = {
    "mb": {"all": [], "DLS": [], "DMS": []},
    "mf": {"all": [], "DLS": [], "DMS": []},
}

regressors = np.array(
    ["response_left", "response_right", "rewarded_incorr", "rewarded_corr"]
)

for sess_id in sess_ids:
    encoder_mb = StrategyEncoder(subj_id, sess_id, strategy_filter="mb")
    encoder_mf = StrategyEncoder(subj_id, sess_id, strategy_filter="mf")

    try:
        encoder_mb.fit_encoder()
        encoder_mf.fit_encoder()
    except RuntimeError:
        continue

    sess_ids_.append(sess_id)

    # get coefs and separate by region as well
    coefs_mb = encoder_mb.encoder_weights
    coefs_mf = encoder_mf.encoder_weights

    tv_idxs_mb = [
        i for i, dm_name in enumerate(encoder_mb.dm_names) if dm_name in regressors
    ]
    tv_idxs_mf = [
        i for i, dm_name in enumerate(encoder_mf.dm_names) if dm_name in regressors
    ]

    coefs_mb = coefs_mb[:, tv_idxs_mb]
    coefs_mf = coefs_mf[:, tv_idxs_mf]

    coefs_sess["mb"]["all"].append(coefs_mb)
    coefs_sess["mf"]["all"].append(coefs_mf)

    coefs_sess["mb"]["DLS"].append(coefs_mb[encoder_mb.reg_idxs["DLS"]])
    coefs_sess["mf"]["DLS"].append(coefs_mf[encoder_mf.reg_idxs["DLS"]])

    coefs_sess["mb"]["DMS"].append(coefs_mb[encoder_mb.reg_idxs["DMS"]])
    coefs_sess["mf"]["DMS"].append(coefs_mf[encoder_mf.reg_idxs["DMS"]])

#### mb

In [ ]:
regr = "response"
val = "left"

coefs_regr = get_coefs_sess_regr(coefs_sess["mb"], reg="all", regr=regr, val=val)
plot_boxplots_sess(
    data_sess=coefs_regr, sess_ids=sess_ids_, data_label=beta_str(regr, val)
)

In [ ]:
regr = "response"
val = "left"

coefs_sess_dict = coefs_to_dict(coefs_sess["mb"], sess_ids_)
plot_ridges_sess(
    data_sess=coefs_sess_dict,
    key=f"{regr}_{val}",
    sess_ids=sess_ids,
    data_label=beta_str(regr, val),
)

#### mf

In [ ]:
regr = "response"
val = "left"

coefs_regr = get_coefs_sess_regr(coefs_sess["mf"], reg="all", regr=regr, val=val)
plot_boxplots_sess(
    data_sess=coefs_regr, sess_ids=sess_ids_, data_label=beta_str(regr, val)
)

In [ ]:
regr = "response"
val = "left"

coefs_sess_dict = coefs_to_dict(coefs_sess["mf"], sess_ids_)
plot_ridges_sess(
    data_sess=coefs_sess_dict,
    key=f"{regr}_{val}",
    sess_ids=sess_ids,
    data_label=beta_str(regr, val),
)